# 重排序（Re-ranking）：交互式实验

本 notebook 把中文镜像站中的 [Cookbook](/cookbooks/rerank_typesafe/) 改写成可以逐格运行、修改输入并观察结果的最小实验。
用 Noul 概率对候选内容进行语义相关性排序。

运行方式与 `../03_架构模式/01_架构模式.ipynb` 一致：有有效的 `TYPESAFE_API_KEY` 时调用真实的
TypeSafe API；没有 Key 或返回 401 时使用内置的离线示例答案。后续代码不区分两种模式，便于先学习
控制流，再切换到真实模型观察概率和置信度。

> 学习提示：先顺序运行全部单元格，再回到“定义 state”或“定义问题”的单元格修改内容，重新运行后面的单元格。
> API Key 只从环境变量读取，不能写进 notebook。


## 0. 准备

### 0.1 安装依赖

In [ ]:
%pip install -q -U typesafe-sdk

### 0.2 创建客户端

In [ ]:
import os
import statistics
import time
from pprint import pprint

from typesafe_sdk import (
    Choice,
    Score,
    Noul,
    TypeSafeClient,
    TypeSafeAuthenticationError,
)

API_KEY = os.environ.get("TYPESAFE_API_KEY", "")
client = TypeSafeClient(api_key=API_KEY, model="jev-latest") if API_KEY else None
print("客户端已创建：模型=jev-latest，Key=", "已配置" if API_KEY else "未配置（将使用离线示例）")


### 0.3 离线响应与统一调用入口

In [ ]:
class _FakeAnswer:
    def __init__(self, type_, **values):
        self.type = type_
        for key, value in values.items():
            setattr(self, key, value)


class _FakeResponse:
    def __init__(self, answers):
        self.answers = answers
        self.nouls = {k: v for k, v in answers.items() if v.type == "noul"}
        self.choices = {k: v for k, v in answers.items() if v.type == "choice"}
        self.scores = {k: v for k, v in answers.items() if v.type == "score"}
        self.model = "jev-latest（离线示例）"
        self.usage = _FakeAnswer("usage", input_tokens=0, output_tokens=0)


class TS:
    offline = False
    _warned = False

    @classmethod
    def call(cls, state, questions, offline_answers):
        if client is None:
            cls.offline = True
            if not cls._warned:
                cls._warned = True
                print("⚠️ 未设置有效 TYPESAFE_API_KEY，以下输出使用内置离线示例。")
            return _FakeResponse(offline_answers)
        try:
            return client.system_one(state, questions)
        except TypeSafeAuthenticationError:
            cls.offline = True
            if not cls._warned:
                cls._warned = True
                print("⚠️ 未设置有效 TYPESAFE_API_KEY，以下输出使用内置离线示例。")
            return _FakeResponse(offline_answers)


def answer_line(name, answer):
    if answer.type == "noul":
        return f"{name}: noul={answer.noul:.2f}"
    if answer.type == "choice":
        return f"{name}: choice={answer.choice} confidence={answer.confidence:.2f}"
    return f"{name}: score={answer.score:.2f} confidence={answer.confidence:.2f}"


print("模式：", "离线示例" if TS.offline else "真实 API（首次调用后确定）")


### 0.4 连通性测试

In [ ]:
if client is None:
    TS.offline = True
    print("⚠️ API Key 未设置，后续单元格使用离线示例。")
else:
    try:
        ping = client.system_one("你好", {"is_greeting": Noul(instructions="这段文字是在打招呼吗？")})
        print("✅ API 连通正常，后续单元格会使用真实结果。")
    except TypeSafeAuthenticationError:
        TS.offline = True
        print("⚠️ API Key 无效，后续单元格使用离线示例。")


## 1. 重排序（Re-ranking）

对每个“查询–候选”对提出一个 Noul 问题，直接按返回概率排序。这里不把概率粗暴地变成固定阈值，
而是保留相对顺序。


### 1.1 定义查询和候选段落

In [ ]:
QUERY = "如何撤销一笔尚未结算的转账？"
CANDIDATES = [
    "你可以在转账详情页点击撤销；已结算的转账需要联系客服。",
    "银行卡挂失后，请在安全中心重新设置登录密码。",
    "退款通常会在五个工作日内原路返回。",
    "如果转账已经结算，收款方需要主动退回资金。",
]
print("查询已定义：候选数=", len(CANDIDATES))


### 1.2 为每个候选构造问题并调用

In [ ]:
scores = []
for index, passage in enumerate(CANDIDATES):
    question = {"relevant": Noul(instructions="这段候选内容是否直接回答用户的问题？")}
    offline_value = [0.93, 0.07, 0.19, 0.78][index]
    response = TS.call({"query": QUERY, "candidate": passage}, question,
                       {"relevant": _FakeAnswer("noul", noul=offline_value)})
    probability = response.nouls["relevant"].noul
    scores.append((probability, passage))
    print(f"候选 {index + 1}: relevance={probability:.2f}  {passage}")


### 1.3 排序并设置展示门槛

In [ ]:
ranked = sorted(scores, key=lambda item: item[0], reverse=True)
for rank, (probability, passage) in enumerate(ranked, 1):
    label = "推荐" if probability >= 0.50 else "低相关"
    print(f"{rank}. {label} {probability:.2f}  {passage}")


观察：排序使用概率的相对大小；`0.50` 这里只是决定是否展示的应用阈值，不改变排名。

## 知识补充
- **社区实测数字**（jev-cookbook 重排序实验，80 条 SciFact 查询）：相对不做精排，Jev 的 nDCG@10 提升 +0.0778，比传统精排还高 +0.0332，且精排耗时与 token 用量显著更低——固定候选集上的语义重排是 Jev 的甜点区。
- **排序 vs 过滤**：本篇用概率**保序**；只想要"相关/不相关"的过滤见 `10_RAG段落分类.ipynb`。两者常连用：先过滤再精排。
- **阈值是应用策略**：0.50 只决定"是否展示"，不改变排名；调阈值不需要重跑模型。

## 小结

这本 notebook 的边界很清楚：TypeSafe 只负责受限、可编程的判断；排序、阈值、分组、重建文本和
函数分派都由 Python 代码完成。修改输入或问题后重新运行，就能观察“模型答案 → 确定性代码”的变化。
